# Attribution Tracking Pipeline

**Purpose:** Interactive audit and live tracking of the full UTM attribution lifecycle — from link click to Supabase `user_attribution` row.

## Attribution Data Flow
```
Inbound Link (UTM params / referrer)
    ↓
Landing Page IIFE (index.html)  →  localStorage("hedge_utm")
    ↓
React captureUtmParams()        →  sessionStorage + localStorage
    ↓
writeAttribution() on signup    →  Supabase: user_attribution
    ↓
This notebook / attribution_tracker.py  →  Audit, diagnose, report
```

### Bugs Found & Fixed (March 2026)
1. **IIFE key mismatch** — `index.html` stored `landing_page` but `utm.ts` read `landing_url` → fixed
2. **t.co not normalized** — social link proxies (`t.co`, `lnkd.in`) stored raw → added `REFERRER_NORMALIZATIONS` map
3. **Email links had no UTM** — license/beta emails linked to hedgedge.info with zero UTM → added UTM tags

---

## 1 — Setup & Imports

In [ ]:
import sys, os
import pandas as pd
from datetime import datetime, timedelta, timezone
from collections import Counter

WORKSPACE = r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge"
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

from dotenv import load_dotenv
load_dotenv(os.path.join(WORKSPACE, ".env"))

from shared.supabase_client import get_supabase
sb = get_supabase(use_service_role=True)

print("[OK] Connected to Supabase")

## 2 — Page Views: UTM Source Breakdown
Pull all `page_views` from Supabase and show which have UTM params vs direct.

In [ ]:
# Pull all page_views
total = sb.table("page_views").select("*", count="exact").execute()
utm_rows = (sb.table("page_views")
    .select("utm_source,utm_medium,utm_campaign,referrer_domain,path,created_at")
    .not_.is_("utm_source", "null")
    .order("created_at", desc=True)
    .execute())

print(f"Total page_views:        {total.count}")
print(f"With UTM source:         {len(utm_rows.data)}")
print(f"Without UTM (direct):    {total.count - len(utm_rows.data)}")

# Source breakdown
sources = Counter(r["utm_source"] for r in utm_rows.data)
mediums = Counter(r["utm_medium"] for r in utm_rows.data if r["utm_medium"])

print(f"\n--- UTM Sources ---")
df_sources = pd.DataFrame(sources.most_common(), columns=["source", "count"])
display(df_sources)

print(f"\n--- UTM Mediums ---")
df_mediums = pd.DataFrame(mediums.most_common(), columns=["medium", "count"])
display(df_mediums)

## 3 — User Attribution Table Audit
Check the `user_attribution` table — what sources are users actually attributed to?

In [ ]:
# Pull user_attribution
ua = sb.table("user_attribution").select("*").order("signed_up_at").execute()
df_ua = pd.DataFrame(ua.data)

print(f"Total attributed users: {len(df_ua)}")
print(f"\n--- Attribution by Source ---")
display(df_ua["utm_source"].fillna("(direct)").value_counts().reset_index())

print(f"\n--- Attribution by Medium ---")
display(df_ua["utm_medium"].fillna("(none)").value_counts().reset_index())

print(f"\n--- Attribution by Campaign ---")
display(df_ua["utm_campaign"].fillna("(none)").value_counts().reset_index())

print(f"\n--- Signup Methods ---")
display(df_ua["signup_method"].value_counts().reset_index())

# Check for backfill artifacts (identical created_at timestamps)
if "created_at" in df_ua.columns:
    ts_counts = df_ua["created_at"].value_counts()
    dupes = ts_counts[ts_counts > 1]
    if len(dupes):
        print(f"\n⚠️  {len(dupes)} timestamp(s) appear on multiple rows (possible backfill):")
        display(dupes)

## 4 — Session Linkage: Do Page Views Match Signups?
Check whether UTM-tagged page_views exist within a 2-hour window before each signup. If not, it means the UTM was lost between landing and signup.

In [ ]:
results = []
for row in ua.data:
    signed_up = row.get("signed_up_at")
    if not signed_up:
        continue
    dt = datetime.fromisoformat(signed_up.replace("+00:00", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    window_start = (dt - timedelta(hours=2)).isoformat()
    window_end = dt.isoformat()

    pv_near = (sb.table("page_views")
        .select("utm_source,utm_medium,path,created_at")
        .gte("created_at", window_start)
        .lte("created_at", window_end)
        .not_.is_("utm_source", "null")
        .execute())

    results.append({
        "user_id": row["user_id"][:8] + "...",
        "signed_up": signed_up[:16],
        "attributed_source": row.get("utm_source") or "(direct)",
        "utm_pageviews_2hr": len(pv_near.data),
        "matched_sources": ", ".join(set(p["utm_source"] for p in pv_near.data)) or "—",
    })

df_linkage = pd.DataFrame(results)
display(df_linkage)

linked = sum(1 for r in results if r["utm_pageviews_2hr"] > 0)
print(f"\n✅ {linked}/{len(results)} users have UTM pageviews within 2hr of signup")
print(f"❌ {len(results) - linked}/{len(results)} users have NO nearby UTM pageviews")

## 5 — Attribution Models: First-Touch, Last-Touch, Linear
Compare how different attribution models credit each marketing channel.

In [ ]:
from collections import defaultdict

rows = ua.data

# First-touch: credit the utm_source
first_touch = defaultdict(int)
for r in rows:
    first_touch[r.get("utm_source") or "direct"] += 1

# Last-touch: credit the utm_medium
last_touch = defaultdict(int)
for r in rows:
    last_touch[r.get("utm_medium") or "none"] += 1

# Linear: equal credit across source + medium + campaign
linear = defaultdict(float)
for r in rows:
    touches = [v for k in ("utm_source", "utm_medium", "utm_campaign") if (v := r.get(k))]
    if not touches:
        touches = ["direct"]
    credit = 1.0 / len(touches)
    for t in touches:
        linear[t] += credit

# Build comparison dataframe
all_channels = sorted(set(list(first_touch) + list(last_touch) + list(linear)))
comparison = []
for ch in all_channels:
    comparison.append({
        "channel": ch,
        "first_touch": first_touch.get(ch, 0),
        "last_touch": last_touch.get(ch, 0),
        "linear": round(linear.get(ch, 0), 2),
    })

df_models = pd.DataFrame(comparison).sort_values("first_touch", ascending=False)
print(f"Attribution Model Comparison ({len(rows)} users)")
display(df_models)

## 6 — Direct Attribution Tracking: Live Pipeline Test
Fire a test pageview with UTM params via Supabase and verify it lands correctly. This validates the full pipeline end-to-end.

In [ ]:
# Direct attribution tracking — insert a test pageview and check it arrives
TEST_UTM = {
    "utm_source": "notebook_test",
    "utm_medium": "qa",
    "utm_campaign": "attribution_pipeline_test",
    "path": "/test-attribution",
    "referrer_domain": "localhost",
    "created_at": datetime.now(timezone.utc).isoformat(),
}

# Insert test pageview
result = sb.table("page_views").insert(TEST_UTM).execute()
test_id = result.data[0]["id"] if result.data else None
print(f"✅ Test pageview inserted (id={test_id})")

# Verify it's retrievable with UTM filters
verify = (sb.table("page_views")
    .select("*")
    .eq("utm_source", "notebook_test")
    .eq("utm_campaign", "attribution_pipeline_test")
    .order("created_at", desc=True)
    .limit(1)
    .execute())

if verify.data:
    print(f"✅ Verified: UTM data round-trips correctly")
    print(f"   source={verify.data[0]['utm_source']}, medium={verify.data[0]['utm_medium']}")
else:
    print(f"❌ FAILED: Could not retrieve test pageview")

# Cleanup test row
if test_id:
    sb.table("page_views").delete().eq("id", test_id).execute()
    print(f"🧹 Cleaned up test row")

## 7 — Conversion Path Analysis
Analyze the most common UTM source → conversion paths and time-to-signup distribution.

In [ ]:
# Build conversion paths from user_attribution
paths = []
for r in ua.data:
    source = r.get("utm_source") or "direct"
    medium = r.get("utm_medium") or "none"
    campaign = r.get("utm_campaign") or "none"
    landed = r.get("landed_at")
    signed = r.get("signed_up_at")

    # Calculate time-to-signup if both timestamps exist
    time_to_signup = None
    if landed and signed:
        try:
            dt_land = datetime.fromisoformat(landed)
            dt_sign = datetime.fromisoformat(signed)
            if dt_land.tzinfo is None:
                dt_land = dt_land.replace(tzinfo=timezone.utc)
            if dt_sign.tzinfo is None:
                dt_sign = dt_sign.replace(tzinfo=timezone.utc)
            time_to_signup = (dt_sign - dt_land).total_seconds() / 60  # minutes
        except Exception:
            pass

    paths.append({
        "channel": f"{source}/{medium}",
        "campaign": campaign,
        "signup_method": r.get("signup_method", "unknown"),
        "time_to_signup_min": round(time_to_signup, 1) if time_to_signup else None,
    })

df_paths = pd.DataFrame(paths)

print("--- Conversion Paths by Channel ---")
display(df_paths["channel"].value_counts().reset_index())

if df_paths["time_to_signup_min"].notna().any():
    print(f"\n--- Time to Signup (minutes) ---")
    display(df_paths["time_to_signup_min"].describe())
else:
    print("\n⚠️  No landed_at timestamps — time-to-signup unavailable (pre-fix signups)")

## 8 — Referrer Normalization Audit
Check how referrers are being categorized. The landing page now normalizes `t.co` → `twitter`, `lnkd.in` → `linkedin`, etc.

In [ ]:
# Check referrer_domain values in page_views — are social proxies still slipping through?
KNOWN_PROXIES = {"t.co", "l.facebook.com", "lnkd.in", "youtu.be"}

ref_rows = (sb.table("page_views")
    .select("referrer_domain,utm_source,utm_medium,created_at")
    .not_.is_("referrer_domain", "null")
    .order("created_at", desc=True)
    .execute())

referrers = Counter(r["referrer_domain"] for r in ref_rows.data)
print(f"--- All Referrer Domains ({len(referrers)} unique) ---")
df_ref = pd.DataFrame(referrers.most_common(), columns=["referrer", "count"])
display(df_ref)

# Check for un-normalized proxies
proxy_leaks = [r for r in ref_rows.data if r["referrer_domain"] in KNOWN_PROXIES
               and r.get("utm_source") == r["referrer_domain"]]
if proxy_leaks:
    print(f"\n⚠️  {len(proxy_leaks)} pageviews still have raw proxy domains as utm_source (pre-fix)")
    for p in proxy_leaks[:5]:
        print(f"   {p['created_at'][:16]}  ref={p['referrer_domain']}  src={p['utm_source']}")
else:
    print(f"\n✅ No un-normalized proxy domains found — referrer normalization working")

## 9 — Full Attribution Summary Report
Generate a complete attribution summary combining all models and diagnostics.

In [ ]:
total_pv = total.count
utm_pv = len(utm_rows.data)
total_users = len(ua.data)
direct_users = sum(1 for r in ua.data if not r.get("utm_source"))
linked_users = sum(1 for r in results if r["utm_pageviews_2hr"] > 0)

print("=" * 60)
print("  ATTRIBUTION PIPELINE HEALTH REPORT")
print("=" * 60)
print(f"\n  Page Views")
print(f"    Total:              {total_pv}")
print(f"    With UTM:           {utm_pv} ({utm_pv/max(total_pv,1)*100:.0f}%)")
print(f"    Direct:             {total_pv - utm_pv}")
print(f"\n  User Attribution")
print(f"    Total users:        {total_users}")
print(f"    With source:        {total_users - direct_users}")
print(f"    Direct/unknown:     {direct_users}")
print(f"    Session-linked:     {linked_users}/{total_users}")
print(f"\n  Top Sources")
for src, cnt in sorted(first_touch.items(), key=lambda x: -x[1])[:5]:
    pct = cnt / max(total_users, 1) * 100
    print(f"    {src:20s} {cnt:3d} ({pct:.0f}%)")
print(f"\n  Pipeline Status")
if direct_users / max(total_users, 1) > 0.5:
    print(f"    ⚠️  >50% users show as direct — UTM capture may still have gaps")
else:
    print(f"    ✅  Attribution pipeline healthy — most users have source data")
print("=" * 60)